# Lecture 13 - Complexity and empirical scaling

**Lecture Topic**: How code scales as the data grows. Order of growth, measuring it, and the budgets that keep it honest. First lecture of Module C.

## Announcements

- The midterm was Monday. Grades and a walkthrough will follow soon.
- **Module C starts today**: the algorithms and data structures behind the code you and your agents write.
- Project reminder: keep going on your M1 work.
- ClassBuzz poll for today: `Z6XV`

Today:
 - Counting the work a piece of code does
 - Order of growth, and the rules for simplifying it
 - Best, worst and average case; Big-O vs. theta
 - Measuring scaling empirically (the log-log slope)
 - Constant factors, memory, and amortized cost
 - Performance budgets, and the cost of the agent itself

# Why complexity

A callback to Lecture 10. I claimed a loop was "far too slow", both agents believed me, and when we finally timed it, it was fine. The lesson was that a performance claim needs a number and a baseline.

Module C is where we make that precise. *Algorithmic complexity* asks one question:

> How does the work an algorithm does grow as the size of its input grows?

Knowing that lets you answer the questions that actually come up on real data:

 - How long will this run on the full file, not the sample?
 - How much memory will it need? Will it even fit?
 - Is this approach feasible at all, or do I need a different one?
 - Which data structure should I reach for?

There are two budgets:

 - **time complexity**: how the runtime grows with the input size.
 - **space complexity**: how the memory use grows with the input size.

## It comes down to counting iterations

Individual operations are fast and roughly constant: an arithmetic operation, a comparison, an `if`. What makes code slow is doing those operations *many times*, so the thing to count is the loops.

Count how many times the innermost work runs as a function of the input size `n`. That count is, essentially, the complexity.

### Question 1

`data` is a list of `n` integers. You run this with `n = 100`, then again with `n = 1000`. Roughly how much longer does the second run take?

```python
total = 0
for i in range(len(data)):
    total += i
```

 - A. The same time
 - B. About 100 times longer
 - C. About 10 times longer
 - D. It is different every time

**C, about 10 times longer.** The loop body runs `n` times, so the work is proportional to `n`. Ten times the data is ten times the work. This is **linear**, O(n).

(An `if` inside the loop would not change that. It still runs at most `n` times, so the order of growth is the same.)

### Question 2

Same setup, but now the loop is nested. With `n = 100` versus `n = 1000`, how much longer does the second run take?

```python
total = 0
for i in range(len(data)):
    for j in range(len(data)):
        total += i * j
```

 - A. The same time
 - B. About 100 times longer
 - C. About 10 times longer
 - D. It is different every time

**B, about 100 times longer.** The inner body runs `n` times for each of the `n` outer passes, so it runs `n * n` times. Ten times the data is a hundred times the work. This is **quadratic**, O(n^2), and it is the most common way code quietly stops scaling.

# Order of growth

We describe that count with its *order of growth*: the shape of the curve, ignoring the details. For an input of size `n`:

| Growth | Name | A typical example |
|---|---|---|
| O(1) | constant | index a list, look up a dict key |
| O(log n) | logarithmic | binary search, halving a number each step |
| O(n) | linear | one pass over the data |
| O(n log n) | linearithmic | a good general-purpose sort |
| O(n^2) | quadratic | compare every pair of items |
| O(2^n) | exponential | try every subset |
| O(n!) | factorial | try every ordering |

The differences are not small. Here are the curves for small `n`, with the y-axis capped so you can see how quickly the fast-growing ones leave the others behind.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = np.arange(1, 21)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(n, np.ones_like(n), label='O(1)')
ax.plot(n, np.log2(n), label='O(log n)')
ax.plot(n, n, label='O(n)')
ax.plot(n, n * np.log2(n), label='O(n log n)')
ax.plot(n, n ** 2.0, label='O(n^2)')
ax.plot(n, 2.0 ** n, label='O(2^n)')
ax.set_ylim(0, 100)
ax.set_xlabel('n')
ax.set_ylabel('operations')
ax.legend(loc='upper left', fontsize=8)

## Three rules for reading off the order

When you work out the count, you simplify it the same way every time:

 - **Drop the constants.** `3n` and `n` are both O(n). Constant multipliers do not change the shape.
 - **The base of a log does not matter.** `log2(n)` and `log10(n)` differ by a constant factor, so both are O(log n).
 - **Keep only the fastest-growing term.** `n^2 + 100n + 500` is O(n^2); for large `n` the `n^2` term dwarfs the rest.

So complexity is deliberately coarse. It tells you the *shape* of the growth rather than the exact runtime.

# Best, worst and average case

The same algorithm can do different amounts of work on different inputs of the same size.

 - **Best case**: the luckiest input. Written with omega, Ω(...).
 - **Worst case**: the unluckiest input. Written with theta on the worst input, and this is what people usually quote.
 - **Average case**: a typical input.

In practice we quote the **worst case**, because it is the guarantee: it is the promise that the algorithm will not do *worse* than this, whatever the data throws at it.

## Big-O vs. theta

A quick note on notation you will see:

 - **theta**, Θ(f), means the growth is *exactly* of that order: bounded above and below by `f`.
 - **Big-O**, O(f), technically means an *upper* bound: "grows no faster than `f`". By that strict reading an O(n) algorithm is also O(n^2).
 - In practice almost everyone uses O(...) to mean the tightest bound, which is the same as theta. We will too.

So when we say an algorithm "is O(n)", we mean linear is the real shape, not just a ceiling.

### Question 3

What is the time complexity of this code, in terms of `n`?

```python
total = 0
for i in range(n):
    j = 0
    while j <= i:
        total += i + j
        j += 1
```

 - A. O(1)
 - B. O(n)
 - C. O(n^2)
 - D. O(log n)
 - E. O(n log n)

**C, O(n^2).** The inner `while` runs `i + 1` times on outer pass `i`, so the total is `1 + 2 + ... + n`, which is `n(n+1)/2`. Drop the constant and the lower-order term and you are left with O(n^2). A triangular loop is still quadratic.

### Question 4

And this one?

```python
j = n
while j > 0:
    print(j)
    j = j // 2
```

 - A. O(1)
 - B. O(n)
 - C. O(n^2)
 - D. O(log n)
 - E. O(n log n)

**D, O(log n).** Each step halves `j`, so the number of steps is how many times you can halve `n` before reaching zero, which is about `log2(n)`. Halving is the signature of a logarithm, and it is why binary search is so cheap.

# Measuring scaling empirically

You do not have to derive the order of growth by staring at the code. You can *measure* it, which is often faster and settles arguments like the one from Lecture 10.

The method: run the code at several input sizes, time each one, and plot time against `n` on **log-log axes**. On those axes a power law `t = c * n^k` becomes a straight line whose **slope is `k`**, the exponent. Linear code gives a slope near 1; quadratic code gives a slope near 2.

Here are two versions of the same task, "does this list contain a duplicate?" One compares every pair; the other remembers what it has seen in a set.

In [ ]:
def has_dup_slow(xs):          # compare every pair: O(n^2)
    for i in range(len(xs)):
        for j in range(i + 1, len(xs)):
            if xs[i] == xs[j]:
                return True
    return False

def has_dup_fast(xs):          # a set remembers what we've seen: O(n)
    seen = set()
    for x in xs:
        if x in seen:
            return True
        seen.add(x)
    return False

We time the worst case (no duplicate, so both have to look at everything) across a range of sizes, then fit the slope on log-log axes.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

def best_time(f, xs, repeats):
    best = float('inf')
    for _ in range(repeats):
        start = time.perf_counter()
        f(xs)
        best = min(best, time.perf_counter() - start)
    return best

sizes = [250, 500, 1000, 2000, 4000]
inputs = [list(range(n)) for n in sizes]          # all unique = worst case

slow_t = [best_time(has_dup_slow, xs, repeats=1) for xs in inputs]
fast_t = [best_time(has_dup_fast, xs, repeats=5) for xs in inputs]

slope_slow = np.polyfit(np.log(sizes), np.log(slow_t), 1)[0]
slope_fast = np.polyfit(np.log(sizes), np.log(fast_t), 1)[0]
print(f"pairwise scan  slope ~ {slope_slow:.2f}  (expect ~2, quadratic)")
print(f"set membership slope ~ {slope_fast:.2f}  (expect ~1, linear)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(sizes, slow_t, 'o-', label=f'pairwise scan (slope {slope_slow:.1f})')
ax.loglog(sizes, fast_t, 's-', label=f'set membership (slope {slope_fast:.1f})')
ax.set_xlabel('n')
ax.set_ylabel('seconds')
ax.legend()

The slopes come out close to 2 and close to 1, which is the quadratic and the linear algorithm showing themselves in the data. Two things worth noticing:

 - The measurement is noisy, especially at small `n` where the timing is dominated by overhead rather than the real work. More, larger sizes give a cleaner slope.
 - On log-log axes the two lines are roughly parallel to the O(n^2) and O(n) references. A constant factor would shift a line up or down but would not change its slope.

This is the practical version of a performance claim: not "it feels slow", but "I measured a slope near 2, so it is quadratic and it will not survive a ten-times-bigger file".

# Constant factors: same shape, very different speed

Order of growth tells you how the cost *grows*. It does not tell you the actual speed, because it throws away the constant factor. In data science that constant can make a big difference.

The classic example: a plain Python loop and a NumPy call are *both* O(n) for summing `n` numbers. But NumPy runs the loop in compiled C over a contiguous array, so its constant factor is tiny.

In [ ]:
import time
import numpy as np

n = 2_000_000
xs = list(range(n))
arr = np.arange(n)

# a little real work per element: square it, then add them up
start = time.perf_counter(); s1 = sum(x * x for x in xs);  t_loop = time.perf_counter() - start
start = time.perf_counter(); s2 = int((arr * arr).sum());  t_vec  = time.perf_counter() - start

print(f"python loop: {t_loop * 1000:6.0f} ms")
print(f"numpy:       {t_vec * 1000:6.1f} ms")
print(f"same answer: {s1 == s2};  numpy is about {t_loop / t_vec:.0f}x faster")

Same complexity, many times the speed, and the gap widens the more real work each element does. This is why "it is only O(n)" is not the end of the discussion, and why vectorising a pandas or NumPy operation can matter even when a loop would, in principle, also finish. Big-O tells you how the cost grows; the constant factor decides whether that growth starts from milliseconds or minutes.

# Space complexity: memory is often the real limit

Time is not the only budget. A computation can be fast and still fall over because an intermediate result does not fit in memory.

The quiet danger is an O(n^2) *space* step hidden inside an otherwise reasonable analysis, like forming all pairwise distances between rows.

In [ ]:
import numpy as np

itemsize = np.dtype(float).itemsize          # 8 bytes per float
for n in [1_000, 10_000, 50_000]:
    gb = itemsize * n * n / 1e9
    print(f"an {n:>6,} x {n:<6,} float matrix needs {gb:8.2f} GB")

A thousand rows is nothing; fifty thousand rows is twenty gigabytes and your laptop is dead. The *time* to build it might be fine, but you never get that far. When an agent proposes a step that materialises an `n`-by-`n` thing, that is the number to check first.

You can measure Python-side peak memory with `tracemalloc` the same way we measured time, but the back-of-the-envelope `n^2` estimate usually catches the disaster before you need to.

# Amortized cost (a taste of next time)

One subtlety for later. Some operations are usually cheap but occasionally expensive, and we average that out over many calls. That average is the *amortized* cost.

Appending to a Python list is **amortized O(1)**: now and then the list has to grow its underlying storage and copy everything, but that happens rarely enough that each append is effectively constant. Building a NumPy array by `np.append` in a loop does *not* get that deal: each call copies the whole array, so the loop is quietly O(n^2).

In [ ]:
import time
import numpy as np

def grow_list(n):
    out = []
    for i in range(n):
        out.append(i)              # amortized O(1) per append
    return out

def grow_nparray(n):
    out = np.array([], dtype=int)
    for i in range(n):
        out = np.append(out, i)    # copies the whole array each time
    return out

for f in [grow_list, grow_nparray]:
    start = time.perf_counter()
    f(40_000)
    print(f"{f.__name__:14s}: {time.perf_counter() - start:.3f} s")

The list is instant; the `np.append` loop is noticeably slow, and it gets quadratically worse. This is exactly why "build a list (or collect the pieces) and convert once at the end" beats growing an array element by element. We will see *why* storage works this way in the next lecture, on arrays and linked lists.

# Performance budgets as acceptance criteria

This closes the loop back to Module B. A verifier only works on things the agent can observe. "Make it faster" is not observable; a number is.

A **performance budget** is just an acceptance criterion with a number in it:

> The full 7-million-row file must process in under 5 seconds and under 2 GB of memory.

Now "fast enough" is a test the agent (or you) can actually run, the same way a correctness test checks the answer.

In [ ]:
import time

budget_s = 5.0
start = time.perf_counter()
has_dup_fast(list(range(2_000_000)))     # the O(n) version
elapsed = time.perf_counter() - start

print(f"ran in {elapsed:.2f} s against a {budget_s:.0f} s budget "
      f"-> {'PASS' if elapsed < budget_s else 'FAIL'}")

Put the budget in the spec and it stops being a matter of opinion. The agent writes a quick timing check, runs it, and reports a pass or a fail with a number, instead of telling you it is "much faster" and hoping you believe it.

# Coda: the complexity of the agent itself

One last place this lens applies, and it is the agent you have been using all term.

The model has no memory between turns, so every turn re-sends the whole conversation so far. Turn `k` sends about `k` turns' worth of tokens, and `1 + 2 + ... + turns` grows like `turns^2`. The cumulative token cost of a long session is **quadratic** in its length, the same triangular sum as Question 3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

turns = np.arange(1, 31)
per_turn = 1000                       # new tokens added each turn
cumulative = np.cumsum(turns * per_turn)   # turn k re-sends ~k turns' worth

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(turns, cumulative / 1e3, 'o-')
ax.set_xlabel('conversation turn')
ax.set_ylabel('cumulative input tokens (thousands)')
ax.set_title('A long agent session costs like turns squared')

This is why the habits from Module B are the right ones: compacting the context and starting a fresh session are not tidiness, they are how you keep a quadratic cost from running away. Caching lowers the price of the repeated prefix, but it does not change the shape of the curve.

# Recap

 - Complexity asks how the work grows with the input size. Count the loops.
 - Simplify: drop constants, ignore the log base, keep the fastest-growing term.
 - Quote the **worst case**; O(...) in practice means the tight bound (theta).
 - You can **measure** scaling: the slope on log-log axes is the exponent.
 - **Constant factors** decide seconds vs. minutes (this is why we vectorise).
 - **Memory** is often the binding budget; watch for `n`-by-`n` intermediates.
 - Make performance a **budget** the agent can test against.
 - Even the agent has a complexity: a long session costs like turns squared.

**Next time (S16): arrays, linked lists and hash tables** - the structures that decide whether an operation is O(1) or O(n) in the first place.